# One-way ANOVA
Sums of squares and F by hand, scipy's `f_oneway`, the multiple-testing risk, and the Titanic age-by-class case study.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Why not several t-tests: the familywise error rate

In [ ]:
for m in [1, 3, 10]:
    print(m, 'tests: P(at least one false alarm) =', round(1 - 0.95 ** m, 3))

## 2. ANOVA by hand: marks of three sections

In [ ]:
groups = [np.array([4, 5, 6]), np.array([6, 7, 8]), np.array([8, 9, 10])]
all_values = np.concatenate(groups)
grand = all_values.mean()
k, N = len(groups), len(all_values)

sst = ((all_values - grand) ** 2).sum()
ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)     # group means vs grand mean
ssw = sum(((g - g.mean()) ** 2).sum() for g in groups)          # values vs own group mean
print('SST', sst, ' SSB', ssb, ' SSW', ssw, ' SSB + SSW', ssb + ssw)

msb, msw = ssb / (k - 1), ssw / (N - k)
F = msb / msw
print('MSB', msb, ' MSW', msw, ' F', F, ' p', round(stats.f.sf(F, k - 1, N - k), 4))
print('critical value', round(stats.f.ppf(0.95, k - 1, N - k), 2))
print(stats.f_oneway(*groups))

## 3. Same means, wider spread

In [ ]:
wide = [[1, 5, 9], [3, 7, 11], [5, 9, 13]]
print(stats.f_oneway(*wide))

## 4. Two groups: F equals t squared

In [ ]:
print('F =', stats.f_oneway([4, 5, 6], [6, 7, 8]).statistic,
      ' t^2 =', round(stats.ttest_ind([4, 5, 6], [6, 7, 8]).statistic ** 2, 6))

## 5. Titanic: age by class

In [ ]:
titanic = pd.read_csv('data/titanic_train.csv')
ages = [titanic.loc[titanic['Pclass'] == c, 'Age'].dropna() for c in (1, 2, 3)]
for c, a in zip((1, 2, 3), ages):
    print(f'class {c}: n = {len(a)}, mean {a.mean():.1f}, sd {a.std():.1f}')
print(stats.f_oneway(*ages))
N = sum(len(a) for a in ages)
print('critical value', round(stats.f.ppf(0.95, 2, N - 3), 2))

In [ ]:
# assumption checks
print('Shapiro p:', [f'{stats.shapiro(a).pvalue:.4f}' for a in ages])
print('Levene', stats.levene(*ages))
# tests that drop an assumption
print('Alexander-Govern', stats.alexandergovern(*ages).pvalue)
print('Kruskal-Wallis', stats.kruskal(*ages).pvalue)

In [ ]:
# which classes differ? (group 0 = class 1)
print(stats.tukey_hsd(*ages))